In [1]:
import os
import re
import json
import time
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv


In [5]:
load_dotenv()

INPUT_FILE  = r"C:\Users\USER\Downloads\alter_medgemma_results.csv"
OUTPUT_FILE = r"C:\Users\USER\Downloads\altermedgemma_results_with_pd.csv"

# A: **[Final|Most Supported] Principal Diagnosis:** text_on_same_line
_PA = re.compile(
    r'\*\*(?:Final\s+|Most Supported\s+)?Principal Diagnosis[:\]]\*\*[:\s]*'
    r'(?!\s*\n\s*\*\*)'
    r'([^\n\*]{3,300})',
    re.IGNORECASE,
)

# B: **[Final|Most Supported] Principal Diagnosis:**\n**text**
_PB = re.compile(
    r'\*\*(?:Final\s+|Most Supported\s+)?Principal Diagnosis[:\]]\*\*[:\s]*\n\s*'
    r'\*\*([^*\n]{3,300})\*\*',
    re.IGNORECASE,
)

# C: ### **Principal Diagnosis** \n **text**
_PC = re.compile(
    r'###\s+\*?\*?Principal Diagnosis\*?\*?\s*\n+\*?\*?([^\n\*]{3,300})',
    re.IGNORECASE,
)

# D: **Principal Diagnosis: text**
_PD = re.compile(
    r'\*\*(?:Final\s+|Most Supported\s+)?Principal Diagnosis:\s*([^*\n]{3,300})\*\*',
    re.IGNORECASE,
)

# E: **bold text** is the most [X] supported principal diagnosis
_PE = re.compile(
    r'\*\*([^*\n]{3,200})\*\*\s+is the most\s+\w*\s*supported principal diagnosis',
    re.IGNORECASE,
)

# F: most [X] supported principal diagnosis is **text** or plain
_PF = re.compile(
    r'most\s+\w*\s*supported principal diagnosis\s+is\s+\*?\*?([^.*\n]{3,200})',
    re.IGNORECASE,
)

# G: medgemma-specific — **...most supported principal diagnosis is:**\n\n**Diagnosis**
# e.g. rows 10 & 13: colon inside bold, diagnosis on next line in its own bold
_PG = re.compile(
    r'most supported principal diagnosis is:\*\*\s*\n+\s*\*\*([^*\n]{3,300})\*\*',
    re.IGNORECASE,
)

PATTERNS = [_PB, _PG, _PC, _PA, _PD, _PE, _PF]

def extract_with_regex(text: str):
    for pat in PATTERNS:
        matches = pat.findall(text)
        if matches:
            return matches[-1].strip().rstrip('*').strip()
    return None

_tests = [
    ("**Principal Diagnosis:** Urosepsis (UTI)",                                 "Urosepsis (UTI)"),
    ("**Most Supported Principal Diagnosis:**\n**Unstable Angina**",             "Unstable Angina"),
    ("### **Principal Diagnosis**\n**Metastatic Breast Cancer**",                "Metastatic Breast Cancer"),
    ("**Principal Diagnosis: Tracheomalacia** (congenital)",                     "Tracheomalacia"),
    ("**Splenic Hematoma** is the most strongly supported principal diagnosis",  "Splenic Hematoma"),
    ("most supported principal diagnosis is **Small Bowel Obstruction**",        "Small Bowel Obstruction"),
    ("the most supported principal diagnosis is Urosepsis.",                     "Urosepsis"),
    ("**Therefore, the most supported principal diagnosis is:**\n\n**Large Subcapsular Splenic Hematoma**",
     "Large Subcapsular Splenic Hematoma"),
]
for text, expected in _tests:
    got = extract_with_regex(text)
    status = "OK" if (got and expected.lower() in got.lower()) else f"FAIL (got: {got!r})"
    print(f"  {status} | expected: {expected!r}")


  OK | expected: 'Urosepsis (UTI)'
  OK | expected: 'Unstable Angina'
  OK | expected: 'Metastatic Breast Cancer'
  OK | expected: 'Tracheomalacia'
  OK | expected: 'Splenic Hematoma'
  OK | expected: 'Small Bowel Obstruction'
  OK | expected: 'Urosepsis'
  OK | expected: 'Large Subcapsular Splenic Hematoma'


In [3]:
API_KEY = os.getenv("DEEPSEEK_API_KEY")
MODEL   = "deepseek-v4-flash"

client = OpenAI(api_key=API_KEY, base_url="https://api.deepseek.com")

EXTRACT_SYSTEM = """You are a clinical NLP extraction assistant.
You will receive the FINAL portion of a model's clinical reasoning output.
Your only job: identify the model's stated principal diagnosis and return it as a short phrase.

Rules:
- Extract exactly what the model concludes, do not infer or add information.
- If the model explicitly names a diagnosis at the end, use that.
- If ambiguous between two candidates, pick the one stated last or most emphatically.
- Strip ICD codes, qualifiers like "(with secondary ...)", footnotes, and coding notes.
- If no clear principal diagnosis is stated, return "UNCLEAR".

Return ONLY valid JSON (no markdown fences):
{"principal_diagnosis": "<short diagnosis phrase>"}"""


def extract_with_llm(tail: str) -> str:
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": EXTRACT_SYSTEM},
                {"role": "user",   "content": tail},
            ],
            max_tokens=128,
            temperature=0.0,
        )
        raw   = resp.choices[0].message.content
        clean = re.sub(r"```json|```", "", raw).strip()
        return json.loads(clean)["principal_diagnosis"]
    except Exception as e:
        return f"ERROR: {e}"


In [6]:
df = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df)} rows from {INPUT_FILE}")

predicted_pds = []
methods       = []

for i, row in df.iterrows():
    text = str(row["final_answer"])

    pd_value = extract_with_regex(text)
    if pd_value:
        method = "regex"
    else:
        tail     = text[-1200:]
        pd_value = extract_with_llm(tail)
        method   = "llm"
        time.sleep(0.3)

    print(f"[{i+1:>2}/{len(df)}] HADM={row['HADM_ID']} cond={row['condition']:<15} [{method}] {str(pd_value)[:90]}")
    predicted_pds.append(pd_value)
    methods.append(method)

df["predicted_pd"] = predicted_pds
df["pd_method"]    = methods

df.to_csv(OUTPUT_FILE, index=False)

n_regex = methods.count("regex")
n_llm   = methods.count("llm")
n_err   = sum(1 for v in predicted_pds if str(v).startswith("ERROR"))
print(f"\nDone → {OUTPUT_FILE}")
print(f"  regex extracted : {n_regex}/{len(df)}")
print(f"  llm  extracted  : {n_llm}/{len(df)}")
print(f"  errors          : {n_err}")
print()
print(df[["HADM_ID", "condition", "ground_truth", "predicted_pd", "pd_method"]].to_string(index=False))


Loaded 35 rows from C:\Users\USER\Downloads\alter_medgemma_results.csv
[ 1/35] HADM=197345 cond=0-shot          [regex] Metastatic Breast Cancer
[ 2/35] HADM=197345 cond=1-shot          [regex] Metastatic Breast Cancer (unspecified, given the lack of specific subtype information beyo
[ 3/35] HADM=197345 cond=counterfactual  [regex] Malignant Pleural Effusion
[ 4/35] HADM=197345 cond=negated_hx      [regex] Ascites
[ 5/35] HADM=197345 cond=negated_ruled_out [regex] Metastatic Breast Cancer
[ 6/35] HADM=197345 cond=negated_hedge   [regex] Metastatic Breast Cancer
[ 7/35] HADM=197345 cond=random_masked   [regex] Malignant Pleural Effusion (likely contributing to L basilar consolidation).
[ 8/35] HADM=176830 cond=0-shot          [regex] Angiodysplasia of the colon.
[ 9/35] HADM=176830 cond=1-shot          [regex] Gastrointestinal Bleed
[10/35] HADM=176830 cond=counterfactual  [regex] Ischemic Colitis.
[11/35] HADM=176830 cond=negated_hx      [regex] Angiodysplasia of the colon.
[12/35] HAD